In [12]:
#!pip install optuna[visualization]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays

In [13]:
df=pd.read_csv('train.csv')
df_=pd.read_csv('test.csv')

In [14]:
df.head()

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0


In [15]:
def transform(df):
    df.date=pd.to_datetime(df.date)
   ## Add Holidays
    extract_country = dict(
        zip(np.sort(df.country.unique()), ["CA", "FI", "IT", "KE", "NO", "SG"]))
    holidays_dict = {
        c: holidays.country_holidays(a, years=range(2010, 2020))
        for c, a in extract_country.items()
    }
    df["is_holiday"] = 0
    for c in holidays_dict:
        df.loc[df.country == c, "is_holiday"] = df.date.isin(holidays_dict[c]).astype(int)


    df["weekday_sv"] = df["date"].dt.strftime("%a").astype("category")
    df["weekday_num"] = df["date"].dt.strftime("%w").astype("category")
    df["day_of_month"] = df["date"].dt.strftime("%d").astype("category")
    df["month_name_sv"] = df["date"].dt.strftime("%b").astype("category")
    df["month_num"] = df["date"].dt.strftime("%m").astype("int")
    df["year_fv"] = df["date"].dt.strftime("%Y").astype("int")
    df["day_number_year"] = df["date"].dt.strftime("%j").astype("int")
    df["week_number_year"] = df["date"].dt.strftime("%W").astype("category")
    df["country"] = df["country"].astype("category")
    df["store"] = df["store"].astype("category")
    df["product"] = df["product"].astype("category")
    df["is_holiday"] = df["is_holiday"].astype("category")
    df["year_sin"] = np.sin(2 * np.pi * df["year_fv"] / 7)
    df["year_cos"] = np.cos(2 * np.pi * df["year_fv"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12)
    return df

df=transform(df)
df_=transform(df_)

In [16]:
df=df[~(df.num_sold.isna())]

In [17]:
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
df_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98550 entries, 0 to 98549
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                98550 non-null  int64         
 1   date              98550 non-null  datetime64[ns]
 2   country           98550 non-null  category      
 3   store             98550 non-null  category      
 4   product           98550 non-null  category      
 5   is_holiday        98550 non-null  category      
 6   weekday_sv        98550 non-null  category      
 7   weekday_num       98550 non-null  category      
 8   day_of_month      98550 non-null  category      
 9   month_name_sv     98550 non-null  category      
 10  month_num         98550 non-null  int32         
 11  year_fv           98550 non-null  int32         
 12  day_number_year   98550 non-null  int32         
 13  week_number_year  98550 non-null  category      
 14  year_sin          9855

In [18]:
for i in ['product','store','country', 'weekday_sv', 'month_name_sv']:
  g=df.groupby(i)['num_sold'].median()
  df[f'impute{i}']=df[i].map(g)
  df_[f'impute{i}']=df_[i].map(g)



C:\Users\dipty\AppData\Local\Temp\ipykernel_21772\3802713132.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g=df.groupby(i)['num_sold'].median()
C:\Users\dipty\AppData\Local\Temp\ipykernel_21772\3802713132.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g=df.groupby(i)['num_sold'].median()
C:\Users\dipty\AppData\Local\Temp\ipykernel_21772\3802713132.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g=df.groupb

In [19]:
for i in ['product','store','country', 'weekday_sv', 'month_name_sv']:
  le=LabelEncoder()
  df[i]=le.fit_transform(df[i])
  df_[i]=le.transform(df_[i])


for i in df.columns[df.dtypes=='category']:
  df[i]=df[i].astype('float')
  df_[i]=df_[i].astype('float')


#df['impute']=(df['imputecountry']+df['imputestore']+df['imputeproduct']+df['imputeis_holiday']+df['imputeweekday_sv']+df['imputeweekday_num']+df['imputeday_of_month']+df['imputemonth_name_sv']+df['imputeweek_number_year'])/9
#df_['impute']=(df_['imputecountry']+df_['imputestore']+df_['imputeproduct']+df_['imputeis_holiday']+df_['imputeweekday_sv']+df_['imputeweekday_num']+df_['imputeday_of_month']+df_['imputemonth_name_sv']+df_['imputeweek_number_year'])/9

#df=df.drop(columns=['imputecountry','imputestore','imputeproduct','imputeis_holiday','imputeweekday_sv','imputeweekday_num','imputeday_of_month','imputemonth_name_sv','imputeweek_number_year'])
#df_=df_.drop(columns=['imputecountry','imputestore','imputeproduct','imputeis_holiday','imputeweekday_sv','imputeweekday_num','imputeday_of_month','imputemonth_name_sv','imputeweek_number_year'])

In [20]:
df.head()

,id,date,country,store,product,num_sold,is_holiday,weekday_sv,weekday_num,day_of_month,month_name_sv,month_num,year_fv,day_number_year,week_number_year,year_sin,year_cos,month_sin,month_cos,imputeproduct,imputestore,imputecountry,imputeweekday_sv,imputemonth_name_sv
1,1,2010-01-01,0,0,1,973.0,1.0,0,5.0,1.0,4,1,2010,1,0.0,0.781831,0.62349,0.5,0.866025,1215.0,380.0,731.0,610.0,614.0
2,2,2010-01-01,0,0,2,906.0,1.0,0,5.0,1.0,4,1,2010,1,0.0,0.781831,0.62349,0.5,0.866025,1007.0,380.0,731.0,610.0,614.0
3,3,2010-01-01,0,0,3,423.0,1.0,0,5.0,1.0,4,1,2010,1,0.0,0.781831,0.62349,0.5,0.866025,537.0,380.0,731.0,610.0,614.0
4,4,2010-01-01,0,0,4,491.0,1.0,0,5.0,1.0,4,1,2010,1,0.0,0.781831,0.62349,0.5,0.866025,617.0,380.0,731.0,610.0,614.0
5,5,2010-01-01,0,2,0,300.0,1.0,0,5.0,1.0,4,1,2010,1,0.0,0.781831,0.62349,0.5,0.866025,192.0,744.0,731.0,610.0,614.0


In [21]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['id','date','num_sold']),np.log(df.num_sold),test_size=.2,random_state=95)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from xgboost import XGBRegressor
model=XGBRegressor(**{'booster': 'dart', 'learning_rate': 0.10615262957548557, 'n_estimators': 475, 'max_depth': 9, 'min_child_weight': 5.995693845679518, 'gamma': 0.002228296977264054, 'subsample': 0.8172700531513919, 'colsample_bytree': 0.6618900867139986, 'lambda': 0.04329348179652168, 'alpha': 0.30043147412831545})
model.fit(X_train,y_train)
y_pred = np.exp(model.predict(X_test))
print("MAPE:",MAPE(np.exp(y_test), y_pred))

In [22]:
#!pip install tensorflow

from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],), kernel_regularizer=l2(0.01)),
    BatchNormalization(),  # Batch Normalization after the Dense layer
    Dropout(0.3),

    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1)  # Output layer for regression
])

# Custom MAPE implementation
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = K.clip(y_true, K.epsilon(), None)  # Avoid division by zero
    return K.mean(K.abs((y_true - y_pred) / y_true)) * 100

# Compile the model
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae', mean_absolute_percentage_error])

# Callbacks for training
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, verbose=1)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=200,  # Use a high number for experimentation; EarlyStopping will halt early if needed
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

c:\Users\dipty\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
5532/5532 ━━━━━━━━━━━━━━━━━━━━ 47s 7ms/step - loss: 8.4862 - mae: 1.7735 - mean_absolute_percentage_error: 33.9702 - val_loss: 0.1771 - val_mae: 0.2082 - val_mean_absolute_percentage_error: 4.8169 - learning_rate: 0.0010
Epoch 2/200
5532/5532 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - loss: 0.6070 - mae: 0.5774 - mean_absolute_percentage_error: 11.0306 - val_loss: 0.1536 - val_mae: 0.2689 - val_mean_absolute_percentage_error: 4.9894 - learning_rate: 0.0010
Epoch 3/200
5532/5532 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - loss: 0.4627 - mae: 0.5069 - mean_absolute_percentage_error: 9.7302 - val_loss: 0.4128 - val_mae: 0.5563 - val_mean_absolute_percentage_error: 11.4440 - learning_rate: 0.0010
Epoch 4/200
5532/5532 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - loss: 0.3880 - mae: 0.4583 - mean_absolute_percentage_error: 8.9891 - val_loss: 0.1021 - val_mae: 0.1928 - val_mean_absolute_percentage_error: 3.9338 - learning_rate: 0.0010
Epoch 5/200
5532/5532 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - loss: 0.327

In [23]:
loss, mae, mape = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (MSE): {loss:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test MAPE: {mape:.4f}")

from sklearn.metrics import mean_absolute_percentage_error as MAPE
# Predictions
y_pred = model.predict(X_test)


# Calculate RMSE
mape = MAPE(y_test, y_pred)
print(f"MAPE: {mape:.4f}")

Test Loss (MSE): 0.0339
Test MAE: 0.1363
Test MAPE: 2.6840
1383/1383 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
MAPE: 0.0268


In [24]:
df_['num_sold']=np.exp(model.predict(df_.drop(columns=['id','date'])))
df_[['id','num_sold']].set_index('id').to_csv('submission.csv')

3080/3080 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step


In [25]:
import lightgbm
from lightgbm import LGBMRegressor


def lgbm_objective(trial):

    lgbm_params = {
        "n_estimators": 2000,
        "subsample": trial.suggest_float("subsample", 0.3, 0.9),
        "min_child_samples": trial.suggest_int("min_child_samples", 60, 100),
        "max_depth": trial.suggest_int("max_depth", 4, 25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.001, 0.1),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.001, 0.1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0)

    }

    lgbm_model = LGBMRegressor(**lgbm_params, random_state=95, verbose=-1)

    lgbm_model.fit(X_train, y_train)
    y_pred = np.exp(lgbm_model.predict(X_test))
    return mean_absolute_percentage_error(np.exp(y_test), y_pred)

In [26]:
#!pip install optuna
#!pip install ipywidgets --upgrade

import optuna

study_LGBM = optuna.create_study(study_name="LGBM_Kaggle", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_LGBM.optimize(lgbm_objective, n_trials=100, show_progress_bar=True)

[I 2025-01-10 10:15:30,030] A new study created in memory with name: LGBM_Kaggle


  0%|          | 0/100 [00:00<?, ?it/s]

In [27]:
print("Best trial:", study_LGBM.best_trial)

Best trial: FrozenTrial(number=61, state=1, values=[4.1885954588271215], datetime_start=datetime.datetime(2025, 1, 10, 11, 3, 29, 767284), datetime_complete=datetime.datetime(2025, 1, 10, 11, 4, 20, 803637), params={'subsample': 0.3971663580657737, 'min_child_samples': 62, 'max_depth': 21, 'learning_rate': 0.09023949324996697, 'lambda_l1': 0.07252855479600676, 'lambda_l2': 0.061631639383395755, 'colsample_bytree': 0.4452455826428544}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'subsample': FloatDistribution(high=0.9, log=False, low=0.3, step=None), 'min_child_samples': IntDistribution(high=100, log=False, low=60, step=1), 'max_depth': IntDistribution(high=25, log=False, low=4, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'lambda_l1': FloatDistribution(high=0.1, log=False, low=0.001, step=None), 'lambda_l2': FloatDistribution(high=0.1, log=False, low=0.001, step=None), 'colsample_bytree': FloatDistribution(high=1.0, l

In [28]:
print("Best parameters:", study_LGBM.best_params)

Best parameters: {'subsample': 0.3971663580657737, 'min_child_samples': 62, 'max_depth': 21, 'learning_rate': 0.09023949324996697, 'lambda_l1': 0.07252855479600676, 'lambda_l2': 0.061631639383395755, 'colsample_bytree': 0.4452455826428544}


In [29]:
lgbm_final = LGBMRegressor(**study_LGBM.best_params,n_estimators= 2000,random_state=95,verbose=-1)
lgbm_final.fit(X_train, y_train)
y_pred = np.exp(lgbm_final.predict(X_test))
print("MAPE:",mean_absolute_percentage_error(np.exp(y_test), y_pred))

MAPE: tf.Tensor(4.1885954588271215, shape=(), dtype=float64)


In [30]:
df_.columns

Index(['id', 'date', 'country', 'store', 'product', 'is_holiday', 'weekday_sv',
       'weekday_num', 'day_of_month', 'month_name_sv', 'month_num', 'year_fv',
       'day_number_year', 'week_number_year', 'year_sin', 'year_cos',
       'month_sin', 'month_cos', 'imputeproduct', 'imputestore',
       'imputecountry', 'imputeweekday_sv', 'imputemonth_name_sv', 'num_sold'],
      dtype='object')

In [31]:
df_['num_sold']=np.exp(lgbm_final.predict(df_.drop(columns=['id','date','num_sold'])))
df_[['id','num_sold']].set_index('id').to_csv('submission2.csv')

In [32]:
import xgboost as xgb

def xgb_objective(trial):
    # Hyperparameter search space
    param = {
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
    }

    # Fit model
    model = xgb.XGBRegressor(**param, random_state=42, use_label_encoder=False)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    # Predict and evaluate
    preds = model.predict(X_test)
    mape = MAPE(y_test, preds)
    return mape

In [33]:
study_XGB = optuna.create_study(direction='minimize')  # Minimize RMSE
study_XGB.optimize(xgb_objective, n_trials=100, show_progress_bar=True)  # Run for 50 trials or 10 minutes

# Best parameters and score
print("Best Trial:")
print(f"  Value: {study_XGB.best_trial.value:.4f}")
print("  Params: ")
for key, value in study_XGB.best_trial.params.items():
    print(f"    {key}: {value}")

  0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:39:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:40:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:45:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\dipty\anaconda3\Lib\site-packages\

[W 2025-01-10 22:04:38,029] Trial 84 failed with parameters: {'booster': 'dart', 'learning_rate': 0.11338171722305883, 'n_estimators': 473, 'max_depth': 10, 'min_child_weight': 4.9625842768121515, 'gamma': 0.008187790189976002, 'subsample': 0.825282305981008, 'colsample_bytree': 0.7532166756129888, 'lambda': 0.008029328000027425, 'alpha': 0.4404298884628788} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\dipty\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\dipty\AppData\Local\Temp\ipykernel_21772\3372814184.py", line 20, in xgb_objective
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
  File "c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\skl

KeyboardInterrupt: 

In [34]:
xgb_final = xgb.XGBRegressor(**study_XGB.best_params,random_state=95,verbose=-1)
xgb_final.fit(X_train, y_train)
y_pred = np.exp(xgb_final.predict(X_test))
print("MAPE:",mean_absolute_percentage_error(np.exp(y_test), y_pred))

c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [22:05:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "verbose" } are not used.

  warnings.warn(smsg, UserWarning)


KeyboardInterrupt: 

In [ ]:
df_['num_sold']=np.exp(xgb_final.predict(df_.drop(columns=['id','date','num_sold'])))
df_[['id','num_sold']].set_index('id').to_csv('submission3.csv')

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances

# Plot optimization history
plot_optimization_history(study_LGBM).show()

# Plot parameter importance
plot_param_importances(study_LGBM).show()

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances

# Plot optimization history
plot_optimization_history(study_XGB).show()

# Plot parameter importance
plot_param_importances(study_XGB).show()

In [ ]:
One,Two,three=pd.read_csv('submission.csv'),pd.read_csv('submission2.csv'),pd.read_csv('submission3.csv')
df_['num_sold']=(One.num_sold+Two.num_sold+three.num_sold)/3
df_[['id','num_sold']].set_index('id').to_csv('submissionall.csv')